In [1]:
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

In [2]:
SEED = 42
rng = np.random.default_rng(SEED)

START_DATE = pd.Timestamp("2026-01-01")
END_DATE = pd.Timestamp("2026-06-30")

NUM_RIDERS = 25_000
NUM_AGENTS = 120
NUM_TRIPS = 120_000

In [3]:
# Create the output directory for the tables if it does not exist

PROJECT_ROOT = Path("__file__").resolve().parents[1]
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
# Function to save generated tables

def save_dataframe(df, filename):
    output_path = RAW_DATA_DIR / filename
    df.to_csv(output_path, index=False)

    print(f"Saved {filename}: {len(df)} rows")

In [5]:
# Table: riders

def generate_riders():
    cities = ["Metroville", "Lakewood", "Riverton", "Hillview"]
    customer_segments = ["Occasional", "Regular", "Frequent"]

    rider_ids = [f"R{str(i).zfill(6)}" for i in range(1, NUM_RIDERS + 1)]
    
    signup_start = pd.Timestamp("2024-01-01")
    signup_days = (START_DATE - signup_start).days
    signup_dates = (
        signup_start
        + pd.to_timedelta(
            rng.integers(
                0,
                signup_days + 1,
                size=NUM_RIDERS,
            ),
            unit="D"
        )
    )

    riders = pd.DataFrame({
        "rider_id": rider_ids,
        "signup_date": signup_dates,
        "home_city": rng.choice(cities, size=NUM_RIDERS, p=[0.35, 0.25, 0.22, 0.18]),
        "customer_segment": rng.choice(customer_segments, size=NUM_RIDERS, p=[0.45, 0.35, 0.20])
    })

    return riders

In [6]:
riders = generate_riders()
save_dataframe(riders, "riders.csv")

Saved riders.csv: 25000 rows


In [7]:
# Table: agents

def generate_agents():
    support_teams = ["General Support", "Payments", "AV Specialist", "Safety"]
    agent_ids = [f"A{str(i).zfill(4)}" for i in range(1, NUM_AGENTS + 1)]
    teams = rng.choice(support_teams, size=NUM_AGENTS, p=[0.50, 0.20, 0.20, 0.10])
    tenure_months = rng.integers(1, 73, size=NUM_AGENTS)

    cost_per_hour = []

    for team in teams:
        if team == "General Support":
            hourly_cost = rng.uniform(18, 24)
        elif team == "Payments":
            hourly_cost = rng.uniform(22, 28)
        elif team == "AV Specialist":
            hourly_cost = rng.uniform(25, 34)
        else:
            hourly_cost = rng.uniform(28, 40)

        cost_per_hour.append(round(hourly_cost, 2))

    agents = pd.DataFrame({
        "agent_id": agent_ids,
        "support_team": teams,
        "tenure_months": tenure_months,
        "cost_per_hour": cost_per_hour,
        "location": rng.choice(["Accra Hub", "Remote"], size=NUM_AGENTS, p=[0.60, 0.40]),
        "active_flag": rng.choice([True, False], size=NUM_AGENTS, p=[0.95, 0.05])
    })

    return agents

In [8]:
agents = generate_agents()
save_dataframe(agents, "agents.csv")

Saved agents.csv: 120 rows


In [9]:
# Table: trips

def generate_trips(riders):
    segment_weights = {
        "Occasional": 1.0,
        "Regular": 2.5,
        "Frequent": 5.0
    }

    av_probability_by_city = {
        "Metroville": 0.30,
        "Lakewood": 0.25,
        "Riverton": 0.50,
        "Hillview": 0.35
    }

    
    # 1. We choose the riders who generate trips
    rider_weights = (riders["customer_segment"].map(segment_weights).to_numpy())
    rider_probabilities = (rider_weights / rider_weights.sum())
    selected_indices = rng.choice(riders.index, size=NUM_TRIPS, replace=True, p=rider_probabilities)

    trip_riders = (riders.loc[
        selected_indices,
        ["rider_id", "home_city", "customer_segment",],
    ].reset_index(drop=True))

    
    # 2. Generate the trip ids
    trip_ids = [f"T{str(i).zfill(7)}" for i in range(1, NUM_TRIPS + 1)]

    
    # 3. Generate trip request timestamps
    generation_end = (END_DATE + pd.Timedelta(days=1) - pd.Timedelta(hours=3))
    total_seconds = int((generation_end - START_DATE).total_seconds())

    requested_at = (START_DATE + pd.to_timedelta(rng.integers(0, total_seconds, size=NUM_TRIPS,), unit="s"))

    
    # 4. Assign each trip a city, assumption is the ride is destined to the rider's home city
    trip_city = (trip_riders["home_city"].to_numpy())

    
    # 5. Determine ride type
    ride_types = []

    for city in trip_city:
        av_probability = (av_probability_by_city[city])
        ride_type = rng.choice(["AV", "Traditional"], p=[av_probability, 1 - av_probability],)
        ride_types.append(ride_type)

    
    # 6. Generate trip distance
    distance_km = rng.gamma(shape=2.3, scale=4.0, size=NUM_TRIPS)
    distance_km = np.clip(distance_km, 0.8, 35,)
    distance_km = np.round(distance_km, 2,)

    
    # 7. Generate trip status
    trip_status = rng.choice(["completed", "cancelled",], size=NUM_TRIPS, p=[0.91, 0.09,],)

    
    # 8. Generate start/ completion/ cancellation times

    started_at = []
    completed_at = []
    cancelled_at = []
    cancellation_stage = []

    for i in range(NUM_TRIPS):
        request_time = requested_at[i]
        status = trip_status[i]
        distance = distance_km[i]

        # Suggest waiting time between 2 to 16 minutes before trip starts
        wait_minutes = rng.integers(2, 16,)
        potential_start = (request_time + pd.Timedelta(minutes=int(wait_minutes)))

        if status == "completed":
            started_at.append(potential_start)
            duration_minutes = (distance/25) * 60
            duration_minutes += rng.normal(5, 4, )
            duration_minutes = max(5, duration_minutes,)

            completion_time = (potential_start + pd.Timedelta(minutes=float(duration_minutes)))
            completed_at.append(completion_time)
            cancelled_at.append(pd.NaT)
            cancellation_stage.append(None)
        else:
            # for trips which were cancelled, we assume
                # 60% cancelled before the trip began and 40% during the trip
            pre_start_cancel = (rng.random() < 0.60)

            if pre_start_cancel:
                started_at.append(pd.NaT)
                completed_at.append(pd.NaT)
                cancel_minutes = (rng.integers(1, 11))
                cancel_time = (request_time + pd.Timedelta(minutes=int(cancel_minutes)))
                cancelled_at.append(cancel_time)
                cancellation_stage.append("pre_start")
            else:
                started_at.append(potential_start)
                completed_at.append(pd.NaT)
                minutes_until_cancel = (rng.integers(3, 31))
                cancel_time = (potential_start + pd.Timedelta(minutes=int(minutes_until_cancel)))
                cancelled_at.append(cancel_time)
                cancellation_stage.append("in_trip")

    
    # 9. Generate fare amount; longer distances should have higher fares
    fare_amount = []

    for i in range(NUM_TRIPS):
        distance = distance_km[i]
        status = trip_status[i]
        stage = cancellation_stage[i]

        expected_fare = (2.50 + (distance * 1.15))
        expected_fare += rng.normal(0, 2.0)
        expected_fare = max(3.0, expected_fare,)

        if status == "completed":
            fare = expected_fare
        elif stage == "pre_start":
            # Simulate charging small fees when rides are cancelled before they start
            if rng.random() < 0.45:
                fare = rng.uniform(2.0, 5.0,)
            else:
                fare = 0.0
        else:
            fare = expected_fare * rng.uniform(0.20, 0.70,)
        
        fare_amount.append(round(fare, 2))


    # 10. Build DataFrame
    trips = pd.DataFrame({
        "trip_id": trip_ids,
        "rider_id": trip_riders["rider_id"],
        "city": trip_city,
        "ride_type": ride_types,
        "requested_at": requested_at,
        "started_at": started_at,
        "completed_at": completed_at,
        "cancelled_at": cancelled_at,
        "trip_status": trip_status,
        "fare_amount": fare_amount,
        "distance_km": distance_km
    })

    for column in ["requested_at", "started_at", "completed_at", "cancelled_at"]:
        trips[column] = (pd.to_datetime(trips[column]).dt.round("s"))
    
    return trips

In [10]:
trips = generate_trips(riders)
save_dataframe(trips, "trips.csv")

Saved trips.csv: 120000 rows


In [11]:
# Table: support_tickets

def generate_primary_support_tickets(trips, agents):
    issue_types = ["pickup_issue", "dropoff_issue", "trip_status_issue", "vehicle_access_issue", "payment_issue", "route_issue", "safety_concern", "lost_item", "cancellation_issue", "other"]

    active_agents = agents[agents["active_flag"] == True]

    tickets = []
    ticket_counter = 1

    for _, trip in trips.iterrows():
        # 1. Determing probability of of creating support ticket based on trip status
        if trip["trip_status"] == "completed":
            if trip["ride_type"] == "AV":
                support_probability = 0.11
            else:
                support_probability = 0.07
        else:
            # when trip was cancelled, probability of creating support ticket is high; higher for AV trips
            if trip["ride_type"] == "AV":
                support_probability = 0.42
            else:
                support_probability = 0.30

        # Ensure not every trip should create a support ticket
        if rng.random() > support_probability:
            continue

        # 2. Generate ticket Id
        ticket_id = (f"TK{str(ticket_counter).zfill(7)}")
        ticket_counter += 1

        # 3. Choose issue type
        if trip["trip_status"] == "cancelled":
            issue_probabilities = {
                "pickup_issue": 0.14,
                "dropoff_issue": 0.02,
                "trip_status_issue": 0.10,
                "vehicle_access_issue": 0.06,
                "payment_issue": 0.10,
                "route_issue": 0.03,
                "safety_concern": 0.05,
                "lost_item": 0.01,
                "cancellation_issue": 0.45,
                "other": 0.04
            }
        elif trip["ride_type"] == "AV":
            issue_probabilities = {
                "pickup_issue": 0.23,
                "dropoff_issue": 0.08,
                "trip_status_issue": 0.14,
                "vehicle_access_issue": 0.18,
                "payment_issue": 0.10,
                "route_issue": 0.09,
                "safety_concern": 0.06,
                "lost_item": 0.04,
                "cancellation_issue": 0.03,
                "other": 0.05
            }
        else:
            issue_probabilities = {
                "pickup_issue": 0.12,
                "dropoff_issue": 0.11,
                "trip_status_issue": 0.10,
                "vehicle_access_issue": 0.01,
                "payment_issue": 0.20,
                "route_issue": 0.14,
                "safety_concern": 0.08,
                "lost_item": 0.12,
                "cancellation_issue": 0.05,
                "other": 0.07
            }
        issue_type = rng.choice(list(issue_probabilities.keys()), p=list(issue_probabilities.values()))

        # 4. Determine internal complexity
        high_complexity_issues = {"safety_concern", "vehicle_access_issue", "trip_status_issue"}
        medium_complexity_issues = {"pickup_issue", "dropoff_issue", "route_issue", "cancellation_issue"}

        if issue_type in high_complexity_issues:
            complexity = rng.choice(["medium", "high"], p=[0.35, 0.65])
        elif issue_type in medium_complexity_issues:
            complexity = rng.choice(["low", "medium", "high"], p=[0.20, 0.60, 0.20])
        else:
            complexity = rng.choice(["low", "medium"], p=[0.65, 0.35])

        # 5. Convert the complexity value into a ticket's priority
        if complexity == "high":
            priority = rng.choice(["high", "urgent"], p=[0.70, 0.30])
        elif complexity == "medium":
            priority = rng.choice(["medium", "high"], p=[0.80, 0.20])
        else:
            priority = "low"

        # 6. Choose the channel the support comes through
        channel = rng.choice(["in_app_chat", "phone", "email"], p=[0.65, 0.20, 0.15])

        # 7. Determine what time the ticket was opened making sure it is after the time the trip was completed/ cancelled
        if trip["trip_status"] == "completed":
            reference_time = trip["completed_at"]
        else:
            reference_time = trip["cancelled_at"]

        # Simulate delay so the ticket was created between a minute or 3 hours after trip completed/cancelled
        delay_minutes = int(rng.integers(1, 181))
        opened_at = (reference_time + pd.Timedelta(minutes=delay_minutes))

        # 8. Assign an active agent to the ticket
        initial_agent_id = rng.choice(active_agents["agent_id"])

        # Populate ticket details
        tickets.append({
            "ticket_id": ticket_id,
            "trip_id": trip["trip_id"],
            "rider_id": trip["rider_id"],
            "parent_ticket_id": None,
            "opened_at": opened_at,
            "resolved_at": pd.NaT,
            "issue_type": issue_type,
            "channel": channel,
            "priority": priority,
            "status": "open",
            "initial_agent_id": initial_agent_id,
            "_complexity": complexity
        })
    
    tickets = pd.DataFrame(tickets)
    return tickets

In [12]:
primary_tickets = generate_primary_support_tickets(trips, agents)

In [13]:
# Table: experiment_assignments

def generate_experiment_assignments(primary_tickets, trips):
    workflow_launch_date = pd.Timestamp("2026-04-01")
    eligible_issue_types = {"pickup_issue", "vehicle_access_issue", "trip_status_issue", "cancellation_issue"}

    # Assign each ticket the ride type
    ticket_context = (
        primary_tickets.merge(
            trips[["trip_id", "ride_type"]], on="trip_id", how="left"
        )
    )

    # Determine eligibility: ticket created after 1st April, is an AV trip, issue type listed above
    eligible_tickets = ticket_context[
        (ticket_context["opened_at"] >= workflow_launch_date) &
        (ticket_context["ride_type"] == "AV") &
        (ticket_context["issue_type"].isin(eligible_issue_types))
    ].copy()

    assignments = []

    for i, (_, ticket) in enumerate(eligible_tickets.iterrows(), start=1):
        experiment_group = rng.choice(["Control", "Treatment"], p=[0.50,0.50])
        assignments.append({
            "assignment_id": (f"EXP{str(i).zfill(6)}"),
            "ticket_id": (ticket["ticket_id"]),
            "experiment_name": ("AV_Guided_Support_v1"),
            "experiment_group": (experiment_group),
            "assigned_at": (ticket["opened_at"]),
            "eligibility_flag": True
        })

    experiment_assignments = (pd.DataFrame(assignments))
    return experiment_assignments

In [14]:
experiment_assignments = generate_experiment_assignments(primary_tickets, trips)

In [15]:
# Table: support_events

def generate_support_events(primary_tickets, experiment_assignments, agents):
    tickets = primary_tickets.copy()

    # Standardize datetime precision
    tickets["opened_at"] = pd.to_datetime(tickets["opened_at"]).astype("datetime64[ns]")
    tickets["resolved_at"] = pd.Series(pd.NaT, index=tickets.index, dtype="datetime64[ns]")
    
    events = []
    event_counter = 1

    # Create a dictionary of ticket and their experiment group: Treatment or Control
    experiment_lookup = (experiment_assignments.set_index("ticket_id")["experiment_group"].to_dict())

    # Get active agents only 
    active_agents = agents[agents["active_flag"] == True].copy()

    # Given a team, select an agent from the team
    def choose_agent(team=None):
        if team is not None:
            candidates = active_agents[active_agents["support_team"] == team]

            if len(candidates) > 0:
                return rng.choice(candidates["agent_id"])

        return rng.choice(active_agents["agent_id"])

    # Assign the escalation team based on the issue_type
    def escalation_team(issue_type):
        if issue_type == "payment_issue":
            return "Payments"
        if issue_type == "safety_concern":
            return "Safety"
        if issue_type in {"pickup_issue", "vehicle_access_issue", "trip_status_issue", "route_issue"}:
            return "AV Specialist"
        return "General Support"

    
    # Generate lifecycle for every ticket
    for index, ticket in tickets.iterrows():

        ticket_id = ticket["ticket_id"]
        opened_at = ticket["opened_at"]
        complexity = ticket["_complexity"]
        issue_type = ticket["issue_type"]
        initial_agent = ticket["initial_agent_id"]
        experiment_group = (experiment_lookup.get(ticket_id, "Not Eligible"))

        # 1. Base escalation probability

        if complexity == "low":
            escalation_probability = 0.05
        elif complexity == "medium":
            escalation_probability = 0.22
        else:
            escalation_probability = 0.55

        # Guided workflow modestly reduces escalation probability
        if experiment_group == "Treatment":
            escalation_probability *= 0.75

        escalated = (rng.random() < escalation_probability)

        
        # 2. Determine number of interactions

        if complexity == "low":
            num_interactions = int(rng.integers(1, 3))
        elif complexity == "medium":
            num_interactions = int(rng.integers(2, 5))
        else:
            num_interactions = int(rng.integers(3, 7))

        # Tickets in Treatment experient group may require fewer exchanges because they will be using guided workflow
        if (experiment_group == "Treatment" and num_interactions > 1 and rng.random() < 0.40):
            num_interactions -= 1

        
        # 3. Assignment event

        assignment_time = (opened_at + pd.Timedelta(minutes=int(rng.integers(1, 11))))

        events.append({
            "event_id": (
                f"EV{str(event_counter).zfill(8)}"
            ),
            "ticket_id": ticket_id,
            "event_time": assignment_time,
            "event_type": "assigned",
            "agent_id": initial_agent,
            "escalation_flag": False,
            "handling_minutes": int(
                rng.integers(2, 7)
            ),
        })
        event_counter += 1

        current_time = assignment_time
        current_agent = initial_agent


        # 4. Interaction events

        for interaction_number in range(num_interactions):

            # Simulate delay in responding from both customer and agent perspective
            if complexity == "low":
                delay_minutes = int(rng.integers(5, 31))
            elif complexity == "medium":
                delay_minutes = int(rng.integers(15, 121))
            else:
                delay_minutes = int(rng.integers(30, 361))

            # Treatment tickets will get a smaller delay period
            if experiment_group == "Treatment":
                delay_minutes = max(3, int(delay_minutes * 0.80))

            current_time += pd.Timedelta(minutes=delay_minutes)

            # Agent reply
            if complexity == "low":
                handling_minutes = int(rng.integers(4, 13))
            elif complexity == "medium":
                handling_minutes = int(rng.integers(8, 21))
            else:
                handling_minutes = int(rng.integers(15, 36))

            events.append({
                "event_id": (
                    f"EV{str(event_counter).zfill(8)}"
                ),
                "ticket_id": ticket_id,
                "event_time": current_time,
                "event_type": "agent_reply",
                "agent_id": current_agent,
                "escalation_flag": False,
                "handling_minutes": (
                    handling_minutes
                ),
            })
            event_counter += 1

            # Some cases require customer response
            if interaction_number < (num_interactions - 1):
                customer_delay = int(rng.integers(5, 181))

                current_time += pd.Timedelta(minutes=customer_delay)

                events.append({
                    "event_id": (
                        f"EV{str(event_counter).zfill(8)}"
                    ),
                    "ticket_id": ticket_id,
                    "event_time": current_time,
                    "event_type": "customer_reply",
                    "agent_id": None,
                    "escalation_flag": False,
                    "handling_minutes": 0,
                })
                event_counter += 1


        # 5. Escalation

        if escalated:
            current_time += pd.Timedelta(
                minutes=int(rng.integers(5, 61)))

            specialist_team = escalation_team(issue_type)
            specialist_agent = choose_agent(specialist_team)

            events.append({
                "event_id": (
                    f"EV{str(event_counter).zfill(8)}"
                ),
                "ticket_id": ticket_id,
                "event_time": current_time,
                "event_type": "escalated",
                "agent_id": specialist_agent,
                "escalation_flag": True,
                "handling_minutes": int(
                    rng.integers(5, 16)
                ),
            })
            event_counter += 1

            current_agent = specialist_agent

            # Specialist performs additional work
            current_time += pd.Timedelta(minutes=int(rng.integers(20, 241)))

            events.append({
                "event_id": (
                    f"EV{str(event_counter).zfill(8)}"
                ),
                "ticket_id": ticket_id,
                "event_time": current_time,
                "event_type": (
                    "specialist_reply"
                ),
                "agent_id": current_agent,
                "escalation_flag": False,
                "handling_minutes": int(
                    rng.integers(10, 31)
                ),
            })
            event_counter += 1

        
        # 6. Final resolution

        current_time += pd.Timedelta(minutes=int(rng.integers(5, 31)))

        events.append({
            "event_id": (
                f"EV{str(event_counter).zfill(8)}"
            ),
            "ticket_id": ticket_id,
            "event_time": current_time,
            "event_type": "resolved",
            "agent_id": current_agent,
            "escalation_flag": False,
            "handling_minutes": int(
                rng.integers(3, 11)
            ),
        })
        event_counter += 1

        # Update ticket itself
        tickets.loc[index, "resolved_at"] = current_time
        tickets.loc[index, "status"] = "resolved"

    support_events = pd.DataFrame(events)
    support_events["event_time"] = pd.to_datetime(support_events["event_time"]).astype("datetime64[ns]")

    return tickets, support_events

In [16]:
updated_tickets, support_events = generate_support_events(primary_tickets, experiment_assignments, agents)

In [17]:
updated_tickets["resolution_minutes"] = (updated_tickets["resolved_at"] - updated_tickets["opened_at"]).dt.total_seconds() / 60

In [18]:
escalated_ticket_ids = set(
    support_events.loc[support_events["escalation_flag"], "ticket_id"]
)

updated_tickets["was_escalated"] = updated_tickets["ticket_id"].isin(escalated_ticket_ids)

In [19]:
def generate_repeat_tickets(updated_tickets, experiment_assignments, agents):
    
    tickets = updated_tickets.copy()
    active_agents = agents[agents["active_flag"] == True]

    experiment_lookup = experiment_assignments.set_index("ticket_id")["experiment_group"].to_dict()

    repeat_tickets = []

    # New repeat tickets should continue numbering from existing ticket
    next_ticket_number = len(tickets) + 1

    for _, ticket in tickets.iterrows():
        complexity = ticket["_complexity"]
        was_escalated = ticket["was_escalated"]
        resolution_minutes = ticket["resolution_minutes"]
        experiment_group = experiment_lookup.get(ticket["ticket_id"], "Not Eligible")

        # 1. Base repeat-contact probability
        if complexity == "low":
            repeat_probability = 0.05
        elif complexity == "medium":
            repeat_probability = 0.12
        else:
            complexity = 0.24


        # 2. Escalation will increase probability
        if was_escalated:
            repeat_probability += 0.08

            
        # 3. Long resolution increases probability
        if resolution_minutes > 360:
            repeat_probability += 0.06


        # 4. Guided workflow increases probability
        if experiment_group == "Treatment":
            repeat_probability *= 0.80

        # Probability should be within 60%
        repeat_probability = min(repeat_probability, 0.60)


        # 5. If random probability is greater than 60%, skip and don't create repeat ticket
        if rng.random() >= repeat_probability:
            continue


        # 6. What time was the repeat ticket created
        hours_until_repeat = rng.uniform(6, 24 * 7)
        repeat_opened_at = ticket["resolved_at"] + pd.Timedelta(hours=float(hours_until_repeat))


        # 7. Give repeat ticket a new ticket id
        repeat_ticket_id = (f"TK{str(next_ticket_number).zfill(7)}")
        next_ticket_number += 1


        # 8. Who is the support agent assigned to the ticket
        initial_agent_id = rng.choice(active_agents["agent_id"])


        # 9. What is the complexity of the ticket
        repeat_complexity = complexity
        
        if (complexity == "low" and rng.random() < 0.25):
            repeat_complexity = "medium"
        elif (complexity == "medium" and rng.random() < 0.20):
            repeat_complexity = "high"


        # 10. Let us set priority for the ticket
        if repeat_complexity == "high":
            priority = rng.choice(["high", "urgent"], p=[0.65, 0.35])
        elif repeat_complexity == "medium":
            priority = rng.choice(["medium", "high"], p=[0.75, 0.25])
        else:
            priority = "low"

        repeat_tickets.append({
            "ticket_id": repeat_ticket_id,
            "trip_id": ticket["trip_id"],
            "rider_id": ticket["rider_id"],
            "parent_ticket_id": ticket["ticket_id"],
            "opened_at": repeat_opened_at,
            "resolved_at": pd.NaT,
            "issue_type": ticket["issue_type"],
            "channel": rng.choice(["in_app_chat", "phone", "email"], p=[0.55, 0.30, 0.15]),
            "priority": priority,
            "status": "open",
            "initial_agent_id": initial_agent_id,
            "_complexity": repeat_complexity
        })

    repeat_tickets = pd.DataFrame(repeat_tickets)

    return repeat_tickets

In [20]:
# Repeat tickets will not be eligible for experiment_assignments
# so we can generate an empty experiment_assignment dataframe when we are creating the support events for the repeated tickets

repeat_tickets = generate_repeat_tickets(updated_tickets, experiment_assignments, agents)
empty_experiment_assignments = experiment_assignments.iloc[0:0]

resolved_repeat_tickets, repeat_events = generate_support_events(repeat_tickets, empty_experiment_assignments, agents)

In [21]:
# Combine original and repeated support ticket/ events so we can have a database of repeated tickets within
all_tickets = pd.concat([updated_tickets, resolved_repeat_tickets], ignore_index=True)
all_support_events = pd.concat([support_events, repeat_events], ignore_index=True)

In [22]:
# Table: Financial_adjustments

def generate_financial_adjustments(all_tickets, trips, all_support_events,):
    adjustments = []
    adjustment_counter = 1

    # Each trip_id and its fare to a dictionary
    trip_lookup = trips.set_index("trip_id")["fare_amount"].to_dict()

    # Extract all tickets that were escalated; refunds/appeasement shd be a little higher
    escalated_ticket_ids = set(all_support_events.loc[all_support_events["escalation_flag"] == True,"ticket_id",])

    # Now process each ticket
    for _, ticket in all_tickets.iterrows():

        ticket_id = ticket["ticket_id"]
        trip_id = ticket["trip_id"]
        issue_type = ticket["issue_type"]
        complexity = ticket["_complexity"]
        fare_amount = trip_lookup.get(trip_id, 0.0)
        was_escalated = ticket_id in escalated_ticket_ids
        is_repeat = pd.notna(ticket["parent_ticket_id"])
        resolution_minutes = (ticket["resolved_at"] - ticket["opened_at"]).total_seconds() / 60

        # 1. For each issue assign the base refund probability
        refund_probability_by_issue = {
            "payment_issue": 0.45,
            "cancellation_issue": 0.40,
            "trip_status_issue": 0.30,
            "pickup_issue": 0.20,
            "dropoff_issue": 0.18,
            "route_issue": 0.15,
            "vehicle_access_issue": 0.18,
            "safety_concern": 0.20,
            "lost_item": 0.05,
            "other": 0.08,
        }
        refund_probability = refund_probability_by_issue[issue_type]

        
        # 2. Increase refund probability is scenarios: 
        # high complexity, was escalated, is repeated, resolved after 6 hours
        # But refund probability should not exceed 75%
        if complexity == "high":
            refund_probability += 0.08
        if was_escalated:
            refund_probability += 0.05
        if is_repeat:
            refund_probability += 0.07
        if resolution_minutes > 360:
            refund_probability += 0.05
        refund_probability = min(refund_probability, 0.75)

        
        # 3. Generate the amount to be refunded
        receives_refund = (fare_amount > 0 and rng.random() < refund_probability)

        if receives_refund:
            if issue_type in {"payment_issue", "cancellation_issue"}:
                refund_fraction = rng.uniform(0.50, 1.00)
            else:
                refund_fraction = rng.uniform(0.20, 0.80)

            refund_amount = round(fare_amount * refund_fraction, 2,)
            created_at = ticket["resolved_at"] + pd.Timedelta(minutes=int(rng.integers(5, 61,)))

            adjustments.append({
                "adjustment_id": (
                    f"ADJ{str(adjustment_counter).zfill(7)}"
                ),
                "ticket_id": ticket_id,
                "trip_id": trip_id,
                "adjustment_type": "refund",
                "amount": refund_amount,
                "reason": issue_type,
                "created_at": created_at,
                "currency": "USD",
            })
            adjustment_counter += 1

        
        # 4. Appeasement-credit probability
        appeasement_probability = 0.03

        if complexity == "high":
            appeasement_probability += 0.10
        if was_escalated:
            appeasement_probability += 0.08
        if is_repeat:
            appeasement_probability += 0.10
        if resolution_minutes > 360:
            appeasement_probability += 0.07

        appeasement_probability = min(appeasement_probability, 0.45)


        # 5. Generate the associated appeasement credit
        if rng.random() < appeasement_probability:

            credit_amount = round(
                rng.uniform(3.0, 20.0), 2)

            created_at = ticket["resolved_at"] + pd.Timedelta(minutes=int(rng.integers(5, 91)))
            
            adjustments.append({
                "adjustment_id": (
                    f"ADJ{str(adjustment_counter).zfill(7)}"
                ),
                "ticket_id": ticket_id,
                "trip_id": trip_id,
                "adjustment_type": "appeasement_credit",
                "amount": credit_amount,
                "reason": "service_recovery",
                "created_at": created_at,
                "currency": "USD",
            })
            adjustment_counter += 1

    financial_adjustments = pd.DataFrame(adjustments)

    return financial_adjustments

In [23]:
financial_adjustments = generate_financial_adjustments(all_tickets, trips, all_support_events)

In [24]:
# Table: generate_customer_feedback

def generate_customer_feedback(all_tickets, experiment_assignments):
    feedback = []
    feedback_counter = 1

    # create a dictionary of ticket ids and experiment group type: Treatment or Control
    experiment_lookup = experiment_assignments.set_index("ticket_id")["experiment_group"].to_dict()

    for _, ticket in all_tickets.iterrows():
        ticket_id = ticket["ticket_id"]
        complexity = ticket["_complexity"]
        was_escalated = ticket["was_escalated"]
        is_repeat = pd.notna(ticket["parent_ticket_id"])
        resolution_minutes = (ticket["resolved_at"] - ticket["opened_at"]).total_seconds() / 60
        experiment_group = experiment_lookup.get(ticket_id, "Not Eligible")

        # 1. Determine the base probability a survey will receive a response
        response_probability = 0.45

        if rng.random() >= response_probability:
            continue

        # 2. Let us assume the rider had a somewhat positive experience with support
        satisfaction_score = 3.5

        # 3. Add or subtract from satisfaction_score based on ticket complexity
        if complexity == "low":
            satisfaction_score += 0.4
        elif complexity == "high":
            satisfaction_score -= 0.5

        # 4. How quickly or otherwise will also add up or subtract from satisfaction score
        if resolution_minutes <= 60:
            satisfaction_score += 0.5
        elif resolution_minutes <= 180:
            satisfaction_score += 0.2
        elif resolution_minutes > 360:
            satisfaction_score -= 0.7

        # 5. An escalated ticket will also reduce the satisfaction_score
        if was_escalated:
            satisfaction_score -= 0.4

        # 6. A repeated ticket will definately impact satisfaction_score
        if is_repeat:
            satisfaction_score -= 0.6

        # 7. A ticket using the guided workflow is expected to be resolved in a smooth manner hence increasing the satisfaction_score
        if experiment_group == "Treatment":
            satisfaction_score += 0.25

        # 8. Let us just add a random value between 0 and 0.65 so same experience does not equal to same CSAT score
        satisfaction_score += rng.normal(0, 0.65)

        # 9. Convert satisfaction_Score to 1 - 5 CSAT scale
        csat_score = int(round(np.clip(satisfaction_score, 1, 5)))

        # 10. Create a new column called feedback based on CSAT score
        if csat_score >= 4:
            feedback_category = "positive"
        elif csat_score == 3:
            feedback_category = "neutral"
        else:
            feedback_category = "negative"

        # 11. Generate time the survey response was submitted 
        submitted_at = (ticket["resolved_at"] + pd.Timedelta(minutes=int(rng.integers(5, 1441))))

        # Populate feedback with details
        feedback.append({
            "feedback_id": (f"FB{str(feedback_counter).zfill(7)}"),
            "ticket_id": ticket_id,
            "rider_id": ticket["rider_id"],
            "csat_score": csat_score,
            "submitted_at": submitted_at,
            "feedback_category": feedback_category
        })
        feedback_counter += 1

    customer_feedback = pd.DataFrame(feedback)

    return customer_feedback

In [25]:
customer_feedback = generate_customer_feedback(all_tickets, experiment_assignments)

In [28]:
# We will inject some data quality problems so we can test data cleaning skills in the project

def inject_data_quality_issues(support_tickets, support_events, financial_adjustments, customer_feedback):
    # 1. Duplicate financial adjustments
    duplicate_adjustments = (
        financial_adjustments.sample(n=min(25, len(financial_adjustments)), random_state=42).copy()
    )

    financial_adjustments_dirty = pd.concat([financial_adjustments, duplicate_adjustments], ignore_index=True)


    # 2. Missing trip_id 
    support_tickets_dirty = support_tickets.copy()
    missing_trip_indices = support_tickets_dirty.sample(n=min(30, len(support_tickets_dirty)), random_state = 43).index
    support_tickets_dirty.loc[missing_trip_indices, "trip_id",] = None
    
    
    # 3. Support events without an associated support ticket
    orphan_events = support_events.sample(n=min(10, len(support_events)), random_state=44).copy()
    orphan_events["ticket_id"] = [
        f"TK_ORPHAN_{i:03d}" 
        for i in range(1, len(orphan_events) + 1)
    ]
    
    support_events_dirty = pd.concat([support_events, orphan_events], ignore_index=True)
    
    
    # 4. Support tickets whose timestamps are incorrect: resolved before it was opened
    bad_timestamp_indices = support_tickets_dirty.sample(n=min(10, len(support_tickets_dirty)), random_state=45).index
    
    support_tickets_dirty.loc[bad_timestamp_indices, "resolved_at", ] = (
        support_tickets_dirty.loc[bad_timestamp_indices, "opened_at"] - pd.Timedelta(hours=2)
    )
    
    
    # 5. Invalid CSAT scores
    customer_feedback_dirty = customer_feedback.copy()
    
    bad_csat_indices = customer_feedback_dirty.sample(n=min(10, len(customer_feedback_dirty)), random_state=46).index
    customer_feedback_dirty.loc[bad_csat_indices, "csat_score",] = 6

    # We need to remove some columns which were only helpful in the data generation process
    columns_to_remove = ["_complexity", "resolution_minutes", "was_escalated"]
    support_tickets_export = support_tickets_dirty.drop(columns=columns_to_remove, errors="ignore")

    return (support_tickets_export, support_events_dirty, financial_adjustments_dirty, customer_feedback_dirty)

In [29]:
(support_tickets_export, support_events_export, financial_adjustments_export, customer_feedback_export) = inject_data_quality_issues(all_tickets, all_support_events, financial_adjustments, customer_feedback)

In [37]:
# Saving dataframe for other 5 tables
save_dataframe(support_tickets_export, "support_tickets.csv")

Saved support_tickets.csv: 14697 rows


In [38]:
save_dataframe(support_events_export, "support_event_tickets.csv")

Saved support_event_tickets.csv: 113021 rows


In [39]:
save_dataframe(financial_adjustments_export, "financial_adjustments.csv")

Saved financial_adjustments.csv: 6106 rows


In [40]:
save_dataframe(customer_feedback_export, "customer_feedback.csv")

Saved customer_feedback.csv: 6760 rows


In [41]:
save_dataframe(experiment_assignments, "experiment_assignments.csv")

Saved experiment_assignments.csv: 1748 rows


In [42]:
print(f"Riders: {len(riders):,}")
print(f"Agents: {len(agents):,}")
print(f"Trips: {len(trips):,}")
print(f"Support Tickets: {len(support_tickets_export):,}")
print(f"Support Events: {len(support_events_export):,}")
print(f"Financial Adjustments: {len(financial_adjustments_export):,}")
print(f"Customer Feedback: {len(customer_feedback_export):,}")
print(f"Experiment Assignments: {len(experiment_assignments):,}")

Riders: 25,000
Agents: 120
Trips: 120,000
Support Tickets: 14,697
Support Events: 113,021
Financial Adjustments: 6,106
Customer Feedback: 6,760
Experiment Assignments: 1,748
